# Study and prepare release dates

We examine `release_date` and replace it with the numeric feature `release_year`. We retain the year as an initial representation of the release period and exclude month and day.

We keep the cleaned training and test copies in memory and record our findings in `reports/release_date_preparation/summary.md`. Raw files and the existing processed CSV files remain unchanged.

## 1. Load the original data

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

DATA_DIR = next(
    (directory for directory in (
        Path.cwd() / "data/raw",
        Path.cwd().parent / "data/raw",
        Path.cwd(),
    ) if all((directory / name).is_file() for name in ("train.csv", "test.csv"))),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("Set DATA_DIR to the folder containing train.csv and test.csv.")
train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")


## 2. Study date validity and precision

We parse dates using the `YYYY-MM-DD` format. We count missing or invalid dates, inspect their range, and measure the proportion dated January 1. A concentration on this date can suggest default month/day values, but does not prove that the dates are approximate.

In [2]:
def parse_release_dates(values):
    return pd.to_datetime(values, format="%Y-%m-%d", errors="coerce")


def summarize_release_dates(datasets):
    rows = []
    for name, frame in datasets.items():
        values = frame["release_date"]
        dates = parse_release_dates(values)
        missing = values.isna() | values.astype("string").str.strip().eq("").fillna(False)
        valid_count = int(dates.notna().sum())
        january_first = int(((dates.dt.month == 1) & (dates.dt.day == 1)).sum())
        rows.append({
            "dataset": name,
            "missing dates": int(missing.sum()),
            "invalid non-missing dates": int((~missing & dates.isna()).sum()),
            "earliest date": dates.min().date().isoformat() if valid_count else None,
            "latest date": dates.max().date().isoformat() if valid_count else None,
            "distinct years": dates.dt.year.nunique(),
            "January 1 tracks": january_first,
            "% January 1 among valid dates": round(100 * january_first / valid_count, 2) if valid_count else 0.0,
        })
    return pd.DataFrame(rows)


def extract_release_year(frame):
    cleaned = frame.copy()
    position = cleaned.columns.get_loc("release_date")
    years = parse_release_dates(cleaned["release_date"]).dt.year.astype("Int16")
    cleaned = cleaned.drop(columns="release_date")
    cleaned.insert(position, "release_year", years)
    return cleaned

release_date_summary = summarize_release_dates({"train": train_raw, "test": test_raw})
display(release_date_summary)


,dataset,missing dates,invalid non-missing dates,earliest date,latest date,distinct years,January 1 tracks,% January 1 among valid dates
0,train,0,0,1956-01-01,2023-12-22,68,40863,11.67
1,test,0,0,1956-01-01,2023-12-22,68,10147,11.59


## 3. Initial decision and transformation

We retain only the year to represent release periods without relying initially on month/day precision. This is a modeling choice whose effect still needs to be assessed with validation ROC AUC.

We replace the original column in both datasets, preserve the other values, and use a nullable integer for the year. Missing or unparseable dates would produce a missing year rather than an invented value.

In [3]:
train_clean = extract_release_year(train_raw)
test_clean = extract_release_year(test_raw)
for original, cleaned in ((train_raw, train_clean), (test_raw, test_clean)):
    assert "release_date" not in cleaned.columns
    assert cleaned["release_year"].dtype == pd.Int16Dtype()
    pd.testing.assert_frame_equal(
        original.drop(columns="release_date"), cleaned.drop(columns="release_year"),
    )
    expected_years = parse_release_dates(original["release_date"]).dt.year.astype("Int16").rename("release_year")
    pd.testing.assert_series_equal(cleaned["release_year"], expected_years)
display(train_clean[["id", "release_year"]].head())
print("Replaced release_date with release_year in both datasets; all other values and row order were preserved.")


Replaced release_date with release_year in both datasets; all other values and row order were preserved.


,id,release_year
0,1,1956
1,2,1956
2,3,1956
3,4,1956
4,5,1956


## 4. Record our decision

In [4]:
report_dir = DATA_DIR.parent.parent / "reports/release_date_preparation"
report_dir.mkdir(parents=True, exist_ok=True)
lines = [
    "# Release date preparation", "",
    "We examined date validity, the range of release dates, and the concentration of dates on January 1 in both original datasets.", "",
    "| Dataset | Missing dates | Invalid non-missing dates | Earliest date | Latest date | Distinct years | January 1 tracks | January 1 (%) |",
    "| --- | ---: | ---: | --- | --- | ---: | ---: | ---: |",
]
for _, row in release_date_summary.iterrows():
    lines.append(
        f"| {row['dataset']} | {row['missing dates']} | {row['invalid non-missing dates']} | "
        f"{row['earliest date']} | {row['latest date']} | {row['distinct years']} | "
        f"{row['January 1 tracks']} | {row['% January 1 among valid dates']:.2f} |"
    )
lines += [
    "", "## Decision", "",
    "We replaced `release_date` with the numeric feature `release_year` in both datasets. "
    "We retained year information to represent musical eras and initially excluded month and day. "
    "The concentration on January 1 suggests that some dates may use default month and day values, "
    "although the data alone does not establish this. We therefore chose a simpler initial representation "
    "without claiming that month and day are necessarily uninformative.", "",
    "We preserved all rows and other values. We stored the year as a nullable integer; missing or unparseable dates "
    "would remain missing rather than receive an invented year. The current data contains no such dates. "
    "This transformation uses neither the target nor statistics learned from the data.", "",
    "We included the same step in `delivery/music_hit_prediction.ipynb`. Its effect on model performance "
    "remains to be evaluated using validation ROC AUC. The cleaned frames remain in memory and raw CSV files are unchanged.", "",
]
(report_dir / "summary.md").write_text("\n".join(lines), encoding="utf-8")
print(f"Report saved to {report_dir / 'summary.md'}")


Report saved to /home/goncalotorcato/ist1109358/CD/Projeto/CD/reports/release_date_preparation/summary.md
